<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ZeroToHeroColabCollection/blob/main/GPT_FROM_SCRATCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-08 03:08:46--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-07-08 03:08:46 (30.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# standard imports
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

In [ ]:
# Check if CUDA is available
print("Is GPU available?", torch.cuda.is_available())

# Print the active GPU device name
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    torch.set_default_device('cuda')

Is GPU available? True
GPU Device Name: Tesla T4


In [ ]:
# HYPERPARAMETERS
BATCH_SIZE = 64
CONTEXT_LENGTH = 64
N_EMBD = 192
N_HEADS = 6
N_LAYER = 6
DROPOUT = 0.3
MAX_ITERS= 5000
LR = 6e-4

In [ ]:
# Opening Dataset
with open('input.txt', 'r') as file:
  text = file.read()

In [ ]:
# Creating Mapping Table from String to Integer and vice versa
chars = sorted(list(set(text)))
stoi = {char: i for i, char in enumerate(chars)}
itos = dict(zip(stoi.values(), stoi.keys()))
vocab_size = len(chars)

In [ ]:
# Creating Decode and Encode Functions
encode = lambda string: list(map(lambda char: stoi[char], list(string)))
decode = lambda array: ''.join(list(map(lambda tok: itos[tok], array)))

In [ ]:
# Tokenizing the Dataset
data = torch.tensor(encode(text))

In [ ]:
# Creating Train/Test Split
n = int(0.9 * data.shape[0])
train_data = data[:n]
val_data = data[n:]

In [ ]:
# Create Batches
def get_batch(data_type):
  data = train_data if data_type == 'train' else val_data
  ix = torch.randint(low = 0, high = (data.shape[0] - CONTEXT_LENGTH), size = (BATCH_SIZE,))
  x = torch.stack(tuple(data[i:CONTEXT_LENGTH+i] for i in ix))
  y = torch.stack(tuple(data[i+1:CONTEXT_LENGTH+i+1] for i in ix))
  return x,y

In [ ]:
# Batches
xb, yb = get_batch('train')

In [ ]:
# Creating Self-Attention Head
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(N_EMBD, head_size)
    self.query = nn.Linear(N_EMBD, head_size)
    self.value = nn.Linear(N_EMBD, head_size)
    self.dropout = nn.Dropout(DROPOUT)
    self.register_buffer('tril', torch.tril(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH)))

  def forward(self, X):
    self.k, self.q, self.v = self.key(X), self.query(X), self.value(X)
    self.wei = (self.k @ self.q.transpose(-2, -1)) * X.shape[1] ** -0.5
    self.wei = self.wei.masked_fill(self.tril[:X.shape[1], :X.shape[1]] == 0, float('-inf'))
    self.wei = F.softmax(self.wei, dim = -1)
    self.wei = self.dropout(self.wei)
    self.out = self.wei @ self.v
    return self.out


In [ ]:
# Multi Head attention time
class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(N_EMBD, N_EMBD)
    self.dropout = nn.Dropout(DROPOUT)

  def forward(self, X):
    self.out = torch.cat([h(X) for h in self.heads], dim=-1)
    self.out = self.proj(self.out)
    self.out = self.dropout(self.out)
    return self.out

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(DROPOUT)
    )

  def forward(self, X):
    self.out = self.net(X)
    return self.out

In [ ]:
# Blocks So We Can Repeat What We Doing
class Block(nn.Module):
  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, X):
    self.X = X
    self.X = self.X + self.sa(self.ln1(self.X))
    self.X = self.X +  self.ffd(self.ln2(self.X))
    return self.X


In [ ]:
# Creating the Model
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.C = nn.Embedding(vocab_size, N_EMBD)
    self.position_embeddings = nn.Embedding(CONTEXT_LENGTH, N_EMBD)
    self.blocks = nn.Sequential(*[Block(N_EMBD, N_HEADS) for _ in range(N_LAYER)])
    self.ln_f = nn.LayerNorm(N_EMBD)
    self.lm_head = nn.Linear(N_EMBD, vocab_size)

  def forward(self, idx, targets = None):

    self.token_emb = self.C(idx)
    self.pos_emb = self.position_embeddings(torch.arange(idx.shape[1]))
    # print('token emb forward ', self.token_emb.shape, 'pos emb forward', self.pos_emb.shape)
    self.x = self.token_emb + self.pos_emb
    self.block_result = self.ln_f(self.blocks(self.x))
    self.logits = self.lm_head(self.block_result)

    # loss
    if targets is None:
      self.loss = None
    else:
      self.loss = F.cross_entropy(self.logits.view(-1, self.logits.shape[-1]), targets.view(-1))

    return self.logits, self.loss

  def generate(self, idx, max_out_tokens):
    self.idx = idx
    for _ in range(max_out_tokens):
      # print('idx generate ', self.idx.shape)
      logits, loss = self(self.idx[:, -CONTEXT_LENGTH:])
      # print('logits generate', self.logits.shape)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim=-1)
      char = torch.multinomial(probs, num_samples=1)
      self.idx = torch.cat((self.idx, char), dim=1)
    return self.idx


In [ ]:
# My Model
model = BigramLanguageModel()
model.train()
logits, loss = model.forward(xb, yb)
smpl = model.generate(torch.zeros((1,1), dtype=torch.long), 1000).tolist()[0]
decode(smpl)

"\nrlJZ?!yD;Ux\nJwvdbVii;zb,LaOM;Qm'&hu3i v\nsaM:wSEPs'PM3B;a!$LwUs\n?'jwBxjM3C\noThJttP$;NH&cBkT Tgtl?TA3VrTsUGGUq&BUBu,eBWEQMY F$dxJ$AzBs$MWL Be$3hN-O?!LhQoLIPfSbgkQ\nkGEtGQr$RIgfTzGsyjk &jNbyBg!VAVzGJk&jaMY!Aj\nmZDL GKJZeIzxOBCY.l3GO$hBhh$3EDJfx3gRWBksk-G-YKtF3aOrG$JxHQM\noWRhW!r$AMYWFFr:CeJSzacMbkg,bR,n?pYzlNYHxOR3TzRUnj-QCdREbo!UHjMXJaFqsMwhVx.,oQVQfAkFjo$rg?TYjzjH-!s!!n:YFo?A:zktDohcZMc$kIwN,KYptfSQlZsTB!jS,'BaKskdTfQuH$-e&3&JG;MfgtmmlFVCt&$hSdz!FJ?ONas!T,QGpDL.mdcoGIsR; KTQMUDWrSC,SGprcBkaY$haQNlUEf3BL YRxht,as,HQfOjvyFozhiXcDjAga3,,W3j.FFNdDJZnm!j?vM!HIZMrIPfxjMM\n''?:JHY?Ff$.LhLUAvoiZ&MUUOZl'?lAjS  k!dFK$yrG&RcBioyT'A 'ag;-fJ;tR?3JtKhUibgc$A3gCGaKk$lDyixwMw&GYI;m:QskQaFvFGcB!TfTfB;Fhh Isl3PtpivFURLDBu3oOu3fEmHQLH pbAHaxJxcOfLZxxAWRuX,CdTWq G$JZeMeI TSGKRACOsB$OaRSV:eDGVbfTSlJzFQhdth$jX.NtvCGBF3xUyKRtPp3FuJJ!m$b?gOKtM!BRr.F'ZeL?jWkh 'ji;?!cMjXTtCRQBl$OgGRfxlRc-eMmzS!hC?QQflhKf hGIMrHO,OD vM3VUxCbYSTQu'i?$HJ\nZbQhY,LdWsZOt&&xZHoGiDB3aoc$onMgze-tqAPWTiVxKQfmj3!MBpbY'L.?'!fM3KJ:&v

In [ ]:
# Creating optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = LR)

In [ ]:
for step in range(MAX_ITERS):
  xb, yb = get_batch('train')
  logits, loss = model.forward(xb, yb)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if step % 100 == 0:
    print(f"{step=}: {loss=}")

print(loss.item())

step=0: loss=tensor(4.2960, device='cuda:0', grad_fn=<NllLossBackward0>)
step=100: loss=tensor(2.4734, device='cuda:0', grad_fn=<NllLossBackward0>)
step=200: loss=tensor(2.3106, device='cuda:0', grad_fn=<NllLossBackward0>)
step=300: loss=tensor(2.1600, device='cuda:0', grad_fn=<NllLossBackward0>)
step=400: loss=tensor(2.0610, device='cuda:0', grad_fn=<NllLossBackward0>)
step=500: loss=tensor(2.0002, device='cuda:0', grad_fn=<NllLossBackward0>)
step=600: loss=tensor(1.9329, device='cuda:0', grad_fn=<NllLossBackward0>)
step=700: loss=tensor(1.8632, device='cuda:0', grad_fn=<NllLossBackward0>)
step=800: loss=tensor(1.8421, device='cuda:0', grad_fn=<NllLossBackward0>)
step=900: loss=tensor(1.8155, device='cuda:0', grad_fn=<NllLossBackward0>)
step=1000: loss=tensor(1.7781, device='cuda:0', grad_fn=<NllLossBackward0>)
step=1100: loss=tensor(1.7230, device='cuda:0', grad_fn=<NllLossBackward0>)
step=1200: loss=tensor(1.7015, device='cuda:0', grad_fn=<NllLossBackward0>)
step=1300: loss=tensor(1

In [ ]:
model.eval()

BigramLanguageModel(
  (C): Embedding(65, 192)
  (position_embeddings): Embedding(64, 192)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=192, out_features=32, bias=True)
            (query): Linear(in_features=192, out_features=32, bias=True)
            (value): Linear(in_features=192, out_features=32, bias=True)
            (dropout): Dropout(p=0.3, inplace=False)
          )
        )
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
      )
      (ffd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=192, out_features=768, bias=True)
          (1): ReLU()
          (2): Linear(in_features=768, out_features=192, bias=True)
          (3): Dropout(p=0.3, inplace=False)
        )
      )
      (ln1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm

In [ ]:
smpl = model.generate(torch.zeros((1,1), dtype=torch.long), 1000).tolist()[0]
print(decode(smpl))


PAULINA:
What should is you have longer you;
And, much them. thou shalt for my heeven,
You shall never bow royal selfice of York,
And which lies the deedsal of dispection,
Intending for eyes not his present
Montableman consul. Rememble sleep or pity to my knee--aught,
Shall lieking to brother fall'd our fance,
And broke hell heaven to you heat you be not
Emple, even royal some to stand; too see the life?

ESCALUS:
Would not did be contrave with him.

Clown:
Would I would beseech my business,
For my comfort, but a prolent stower her poor joy;
But thou shalt change a that was faith put your life.

First TNurth, that aparish on you, I am an old
Hath sounded cause my partience to Edward's coal.

CARLIOLANUS:
Take all, how me shall not shall.
When help unthese mase.
So in Edward's cast speak despate.

WARWICK:
We shall poor to such good Help, this own. Thou, brother?

CLARENCE:
The mistricians? how of, speak, the despected? and I, that came
their France. Romeo, God, with one paint me;
He w

In [ ]:
@torch.no_grad()
def eval_loss(data_type):
  losses = []
  for step in range(1000):
    xb, yb = get_batch(data_type)
    logits, loss = model.forward(xb, yb)
    losses.append(loss)
  print("True Loss", torch.tensor(losses).mean())

In [ ]:
eval_loss('train'), eval_loss('test')

True Loss tensor(1.3546, device='cuda:0')
True Loss tensor(1.5766, device='cuda:0')


(None, None)

In [ ]:
# Chat with Shakespeare fr fr
prompt = input('User: ')
custom_filter = lambda char: char in chars
filtered_prompt = ''.join(list(filter(custom_filter, list(prompt))))
tokenized_prompt = encode(filtered_prompt)
response = model.generate(torch.tensor(tokenized_prompt, dtype=torch.long).view(-1,1), 1000).tolist()[0]
print(f"Bootlegged AI Shakespeare: {decode(response)}")